# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Version: {metadata.get('version', 'N/A')}")
print(f"Published: {metadata.get('datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets, their fields, and IDs. In Croissant, entities are referenced by their `@id`. We'll enumerate the record sets and fields, displaying each one's `@id` to guide further data extraction.

**Note:** The dataset may include multiple record sets, each with its fields. We'll extract this structure from the dataset metadata.

In [ ]:
# List all record sets and their fields using their @id

record_sets = dataset.metadata.record_sets

if not record_sets:
    print("No record sets found in the dataset metadata.")
else:
    for rs in record_sets:
        print(f"Record Set: {rs['@id']} (name: {rs.get('name', 'N/A')})")
        fields = rs.get('fields', [])
        for f in fields:
            print(f"  Field: {f['@id']} (name: {f.get('name', 'N/A')})")

# You can also get columns for each record set's fields if available:
        columns = rs.get('columns', [])
        if columns:
            print("  Columns:")
            for col in columns:
                print(f"    Column {col['@id']} (name: {col.get('name', 'N/A')})")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

We reference record sets and fields by their `@id` as established previously.

In [ ]:
# Extract data from the main record sets
# Dynamically find all record set @id values

record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records_gen = dataset.records(record_set=record_set_id)
        records_list = list(records_gen)
        df = pd.DataFrame(records_list)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set @id: {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Show the columns (fields) for the first record set, by @id
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    print(f"Columns in record set {first_record_set_id}: {dataframes[first_record_set_id].columns.tolist()}")
    dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing numeric fields, and grouping. Use record set and field `@id`s as referenced above.

***Note:*** For demonstration, select one record set and some likely numeric and grouping fields based on the overview. Please replace `<numeric_field_id>` and `<group_field_id>` with actual `@id`s as listed previously.

In [ ]:
# Example: Use the first record set and select GAD-7 score and participant village as grouping
first_record_set_id = record_set_ids[0] if record_set_ids else None

if first_record_set_id is not None:
    df = dataframes[first_record_set_id]
    # Manually set likely field @id for GAD-7 score and village, update as needed
    numeric_field_id = None
    group_field_id = None

    # Try to autodetect GAD-7, PHQ-9, PSQ or similar fields
    for col in df.columns:
        if 'GAD' in col or 'gad' in col or 'score' in col:
            numeric_field_id = col
            break

    # Try to find a grouping column (e.g., village)
    for col in df.columns:
        if 'village' in col.lower():
            group_field_id = col
            break

    print(f"Selected numeric field: {numeric_field_id}")
    print(f"Selected grouping field: {group_field_id}")

    if numeric_field_id:
        # Filter records (e.g., GAD-7 score > 10)
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by village (or other selected grouping field)
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field detected in the loaded record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here we use matplotlib to plot the normalized GAD-7 scores or group averages.

In [ ]:
# Visualization example
import matplotlib.pyplot as plt

if first_record_set_id and numeric_field_id:
    # Histogram of normalized GAD-7 score
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col in filtered_df.columns:
        plt.figure(figsize=(8,5))
        filtered_df[norm_col].hist(bins=20)
        plt.xlabel('Normalized GAD-7 Score')
        plt.ylabel('Count')
        plt.title('Distribution of Normalized GAD-7 Scores (filtered >10)')
        plt.show()

    # Bar chart by grouped average, if available
    if group_field_id and 'grouped_df' in locals():
        plt.figure(figsize=(10,5))
        plt.bar(grouped_df[group_field_id], grouped_df[numeric_field_id])
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load, explore, and analyze the FAIR² Mental Health Survey Dataset from Kilifi, using Croissant's `@id` references for all entities. We:
- Inspected metadata and record sets by their `@id`.
- Loaded records from each set and analyzed key numeric variables (e.g., GAD-7 score).
- Filtered, normalized, grouped, and visualized the data.

This workflow provides a foundation for further analysis, e.g., studying demographic effects or integrating additional mental health metrics.